In [1]:
import torch

from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from transformers import BitsAndBytesConfig

## Load & Preprocess

In [2]:
config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [3]:
id2label = {0: "Win A", 1: "Tie", 2: "Win B"}
label2id = {v:k for k, v in id2label.items()}

model_string = "Qwen/Qwen3-0.6B"
model = AutoModelForSequenceClassification.from_pretrained(
    model_string, num_labels=3, id2label=id2label, label2id=label2id,
    quantization_config=config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_string)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-0.6B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# load model with peft
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM"
)

model.add_adapter(peft_config)

In [5]:
dataset = load_dataset("csv", data_files="data/train.csv")["train"]

In [6]:
def preprocess_input(batch):
    """Unifies the prompts and responses into a single sentence and creates a single integer label from the label columns."""
    prompts = []
    for prompt, resp_a, resp_b in zip(batch["prompt"], batch["response_a"], batch["response_b"]):
        prompts.append(f"<prompt>{prompt}\n\n<answer1>{resp_a}\n\n<answer2>{resp_b}")

    labels = []
    for win_a, win_b in zip(batch["winner_model_a"], batch["winner_model_b"]):
        label = 0 if win_a else 2 if win_b else 1
        labels.append(label)
    return {**tokenizer(prompts, truncation=True), "labels": labels}

In [7]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Training

- Try to check what influence certain aspects have on run time
- Run training with some configuration on increasing dataset size and log the train time

In [8]:
def get_runtime(training_arguments):
    """Instantiates the trainer and lets it perform training. Returns the runtime."""
    trainer = Trainer(
        model=model,
        args=training_arguments,
        train_dataset=preprocessed,
        data_collator=data_collator
    )
    summary = trainer.train()
    return summary.metrics["train_runtime"]

### Batch Size

In [ ]:
train_store, ds_size_store, type_store = [], [], []
for dataset_size in [1000, 2000, 4000, 8000]:
    subset = dataset.select(range(dataset_size))
    preprocessed = subset.map(preprocess_input, batched=True)
    for batch_size in [1, 2, 4, 8, 16, 32, 64]:
        training_args = TrainingArguments(
            per_device_train_batch_size=batch_size,
            num_train_epochs=1,
            eval_strategy="no",
        )
        train_time = get_runtime(training_args)

        train_store.append(train_time)
        ds_size_store.append(dataset_size)
        batch_size_store.append(batch_size)

### Mixed Precision

In [ ]:
train_store, ds_size_store, type_store = [], [], []
for dataset_size in [8000, 16000, 32000]:
    subset = dataset.select(range(dataset_size))
    preprocessed = subset.map(preprocess_input, batched=True)

    # Normal training with fp32
    training_args = TrainingArguments(
        per_device_train_batch_size=32,
        num_train_epochs=1,
        disable_tqdm=True
    )
    train_time = get_runtime(training_args)

    train_store.append(train_time)
    ds_size_store.append(dataset_size)
    type_store.append("fp32")
    torch.cuda.empty_cache()

    # fp16
    training_args = TrainingArguments(
        per_device_train_batch_size=64,
        num_train_epochs=1,
        fp16=True,
        disable_tqdm=True
    )
    train_time = get_runtime(training_args)

    train_store.append(train_time)
    ds_size_store.append(dataset_size)
    type_store.append("fp16")
    torch.cuda.empty_cache()

    # bf16
    training_args = TrainingArguments(
        per_device_train_batch_size=64,
        num_train_epochs=1,
        bf16=True,
        disable_tqdm=True
    )
    train_time = get_runtime(training_args)

    train_store.append(train_time)
    ds_size_store.append(dataset_size)
    type_store.append("bf16")
    torch.cuda.empty_cache()

### Other Speedup

In [ ]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

train_store, ds_size_store, type_store = [], [], []
for dataset_size in [4000, 8000, 16000]:
    subset = dataset.select(range(dataset_size))
    preprocessed = subset.map(preprocess_input, batched=True)

    # Optimizer
    training_args = TrainingArguments(
        per_device_train_batch_size=64,
        num_train_epochs=1,
        bf16=True,
        disable_tqdm=True,
        optim="adamw_bnb_8bit",
    )
    train_time = get_runtime(training_args)

    train_store.append(train_time)
    ds_size_store.append(dataset_size)
    type_store.append("peft_optim")
    torch.cuda.empty_cache()

    # Pinned memory
    training_args = TrainingArguments(
        per_device_train_batch_size=64,
        num_train_epochs=1,
        bf16=True,
        disable_tqdm=True,
        dataloader_pin_memory=True,
        dataloader_num_workers=4,
    )
    train_time = get_runtime(training_args)

    train_store.append(train_time)
    ds_size_store.append(dataset_size)
    type_store.append("peft_pinned_mem")
    torch.cuda.empty_cache()

    # Torch Compile
    training_args = TrainingArguments(
        per_device_train_batch_size=64,
        num_train_epochs=1,
        bf16=True,
        disable_tqdm=True,
        torch_compile=True,
        torch_compile_backend="inductor"
    )
    train_time = get_runtime(training_args)

    train_store.append(train_time)
    ds_size_store.append(dataset_size)
    type_store.append("peft_compile")
    torch.cuda.empty_cache()

### Train with bitsandbytes and larger model

In [9]:
train_store, ds_size_store, type_store = [], [], []
for dataset_size in [4000, 8000, 16000]:
    subset = dataset.select(range(dataset_size))
    preprocessed = subset.map(preprocess_input, batched=True)

    # Optimizer
    training_args = TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        gradient_checkpointing=True,
        num_train_epochs=1,
        fp16=True,
        optim="adamw_bnb_8bit",
    )
    train_time = get_runtime(training_args)

    train_store.append(train_time)
    ds_size_store.append(dataset_size)
    type_store.append("peft_optim")
    torch.cuda.empty_cache()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
500,4.542500
1000,4.269400


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Step,Training Loss


KeyboardInterrupt: 

In [ ]:
import pandas as pd

# train_store, ds_size_store, batch_size_store
runtime_df = pd.DataFrame({"dataset_size": ds_size_store, "type": type_store, "runtime": train_store})
runtime_df.to_csv("runtime-tests/other-speedup.csv")

In [ ]:
import seaborn as sns

sns.set_theme(style="whitegrid", palette="pastel")
runtime_plot = sns.lineplot(runtime_df, x="dataset_size", y="runtime", hue="type", style="type")
fig = runtime_plot.get_figure()
fig.suptitle("Other Speedup Experiment")
fig.savefig("runtime-tests/other-speedup.png", dpi=300)

## Inference

In [ ]:
import pandas as pd
from transformers import Pipeline
from transformers.pipelines.pt_utils import KeyDataset

In [ ]:
# the model gets loaded automatically after train
# only call this if no training was conducted
model = AutoModelForSequenceClassification.from_pretrained("distilbert/checkpoint-3234/")

tokenizer = AutoTokenizer.from_pretrained("distilbert/checkpoint-3234/")

In [ ]:
def preprocess_for_inf(batch):
    """Unifies the prompts and responses into a single sentence and creates a single integer label from the label columns."""
    prompts = []
    for prompt, resp_a, resp_b in zip(batch["prompt"], batch["response_a"], batch["response_b"]):
        prompts.append(f"<prompt>{prompt}\n\n<answer1>{resp_a}\n\n<answer2>{resp_b}")

    return {**tokenizer(prompts, truncation=True), "text": prompts}

In [ ]:
test_data = load_dataset("csv", data_files={"test": "data/test.csv"})["test"]
test_data = test_data.map(preprocess_for_inf, batched=True)

In [ ]:
class SoftmaxPipeline(Pipeline):
    def preprocess(self, inputs, **kwargs):
        return self.tokenizer(inputs, return_tensors="pt", truncation=True)

    def _forward(self, model_inputs, **kwargs):
        outputs = self.model(**model_inputs)
        return outputs

    def postprocess(self, model_outputs, **kwargs):
        return model_outputs["logits"].softmax(dim=-1)

    def _sanitize_parameters(self, **kwargs):
        return {}, {}, {}

pipe = SoftmaxPipeline(task="text-classification", model=model, tokenizer=tokenizer, batch_size=4)

In [ ]:
from collections import defaultdict

test_result = defaultdict(list)
for sample_id, out in zip(test_data["id"], pipe(KeyDataset(test_data, "text"), batch_size=8)):
    probs = out[0].tolist()
    test_result["id"].append(sample_id)
    test_result["winner_model_a"].append(probs[0])
    test_result["winner_model_b"].append(probs[2])
    test_result["tie"].append(probs[1])

test_result = pd.DataFrame(test_result)
test_result.to_csv("submission.csv", index=False)